# 優化實驗視覺化 Notebook

1. **3D Pareto 前沿** - 三維散點圖顯示準確率、GPU、延遲的權衡
2. **Pareto 解詳細表格** - 每個 Pareto 解的配置和指標
3. **All Trials 詳細表格** - 所有試驗（包括失敗/被剪枝）的完整資訊
4. **量化方法比較** - 各方法的平均表現和標準差
5. **平行座標圖** - 同時比較所有三個目標的多維視覺化
6. **權衡散點圖矩陣** - 2x2 矩陣顯示目標之間的兩兩關係
7. **資料集詳細結果** - 推薦配置在各資料集的表現

## 🚀 使用方法

1. 運行前面的 cells 掃描可用實驗
2. 在「選擇要視覺化的實驗」section 修改 `SELECTED_EXPERIMENT` 變數
3. 依序執行所有 cells

In [13]:
import os
import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from typing import Dict, List, Any
import numpy as np

# 設定結果資料夾路徑
RESULTS_DIR = Path('.')
print(f"Results directory: {RESULTS_DIR.absolute()}")

Results directory: /home/claire/Documents/Green_AI/results


## 1. 掃描可用的優化實驗

In [4]:
def scan_optimization_experiments():
    """掃描所有優化實驗"""
    opt_dir = RESULTS_DIR / 'optimization'
    experiments = []
    
    if opt_dir.exists():
        for item in opt_dir.iterdir():
            if item.is_dir() and (item / 'summary.json').exists():
                experiments.append(item.name)
    
    return sorted(experiments)

# 掃描實驗
available_experiments = scan_optimization_experiments()
print("\n=== 可用的優化實驗 ===")
for i, exp in enumerate(available_experiments, 1):
    print(f"{i}. {exp}")

if not available_experiments:
    print("未找到優化實驗結果")


=== 可用的優化實驗 ===
1. gemma-2-2b-it-multiobjective-opt_20260102_205910
2. gemma-2-2b-it-multiobjective-opt_20260108_224919
3. quick-mo-test_20251127_165523
4. quick-mo-test_20251130_161230
5. quick-mo-test_20251202_221816
6. quick-mo-test_20251203_161824


## 2. 選擇要視覺化的實驗

**修改下面的變數來選擇實驗：**

In [5]:
# ========================================
# 在這裡選擇要視覺化的實驗
# ========================================

# 方法 1: 直接指定實驗名稱
SELECTED_EXPERIMENT = "gemma-2-2b-it-multiobjective-opt_20260108_224919"

# 方法 2: 或使用索引選擇（從上面的列表選擇，索引從 0 開始）
# SELECTED_EXPERIMENT = available_experiments[0]  # 選擇第一個

# 方法 3: 或選擇最新的實驗
# SELECTED_EXPERIMENT = available_experiments[-1]  # 選擇最後一個（最新）

print(f"\n選擇的實驗: {SELECTED_EXPERIMENT}")


選擇的實驗: gemma-2-2b-it-multiobjective-opt_20260108_224919


## 3. 載入實驗資料

In [6]:
def load_optimization_results(run_name: str) -> Dict:
    """載入優化實驗結果"""
    run_dir = RESULTS_DIR / 'optimization' / run_name
    results = {}
    
    files = ['summary.json', 'pareto_frontier.json', 'all_trials.json']
    for file in files:
        file_path = run_dir / file
        if file_path.exists():
            with open(file_path, 'r') as f:
                results[file.replace('.json', '')] = json.load(f)
        else:
            print(f"警告: {file} 不存在")
    
    return results

# 載入資料
results = load_optimization_results(SELECTED_EXPERIMENT)

if not results:
    raise ValueError(f"無法載入實驗資料: {SELECTED_EXPERIMENT}")

summary = results.get('summary', {})
pareto = results.get('pareto_frontier', {})
all_trials = results.get('all_trials', {})

print("\n✓ 資料載入成功")


✓ 資料載入成功


## 3.1 輔助函數：格式化配置信息

In [7]:
def format_config(config: Dict) -> str:
    """格式化配置為易讀的字符串（用於 hover 顯示）"""
    if not config:
        return "N/A"
    
    # 排除不重要的字段
    exclude_keys = {'modules_to_not_convert'}
    
    lines = []
    for key, value in config.items():
        if key not in exclude_keys:
            # 簡化鍵名
            display_key = key.replace('bnb_4bit_', '').replace('gptq_', '').replace('awq_', '')
            lines.append(f"{display_key}: {value}")
    
    return "<br>".join(lines)

def format_config_short(config: Dict) -> str:
    """格式化配置為簡短字符串（顯示主要參數）"""
    if not config:
        return "N/A"
    
    method = config.get('method', 'unknown')
    
    # 根據方法提取關鍵參數
    if method == 'gptq':
        bits = config.get('bits', '?')
        group_size = config.get('group_size', '?')
        return f"{method} (bits={bits}, gs={group_size})"
    elif method == 'awq':
        w_bit = config.get('w_bit', '?')
        q_group_size = config.get('q_group_size', '?')
        return f"{method} (w_bit={w_bit}, gs={q_group_size})"
    elif method == 'bnb':
        bits = config.get('bits', '?')
        quant_type = config.get('bnb_4bit_quant_type', 'fp4')
        return f"{method} (bits={bits}, {quant_type})"
    else:
        return method

print("✓ 配置格式化函數已定義")

✓ 配置格式化函數已定義


## 4. 實驗摘要資訊

In [8]:
# 顯示實驗摘要
print("\n" + "="*70)
print("實驗摘要資訊")
print("="*70)

print(f"\n實驗名稱: {summary.get('experiment_name', 'N/A')}")
print(f"時間戳記: {summary.get('timestamp', 'N/A')}")

opt_info = summary.get('optimization', {})
print(f"\n總試驗數: {opt_info.get('total_trials', 'N/A')}")
print(f"Pareto 解數量: {opt_info.get('pareto_solutions', 'N/A')}")
print(f"滿足目標解數: {opt_info.get('satisfying_solutions', 'N/A')}")
print(f"優化器: {opt_info.get('optimizer', 'N/A')}")

# 基準模型資訊
baseline = summary.get('baseline', {})
print(f"\n--- 基準模型 ---")
print(f"模型: {baseline.get('model', 'N/A')}")
print(f"準確率: {baseline.get('accuracy', 0)*100:.2f}%")
print(f"GPU 峰值: {baseline.get('gpu_peak_mb', 0):.0f} MB")
print(f"平均延遲: {baseline.get('avg_latency_ms', 0):.2f} ms")

# 優化目標
targets = summary.get('targets', {})
print(f"\n--- 優化目標 ---")
print(f"準確率最小值: {targets.get('accuracy_min', 0)*100:+.1f}%")
print(f"GPU 峰值最大值: {targets.get('gpu_peak_max', 0)*100:+.1f}%")
print(f"延遲最大值: {targets.get('latency_max', 0)*100:+.1f}%")

# 推薦配置
recommended = summary.get('recommended', {})
if recommended:
    print(f"\n--- 推薦配置 ---")
    print(f"方法: {recommended.get('method', 'N/A')}")
    
    config = recommended.get('config', {})
    print(f"\n配置詳情:")
    for key, value in config.items():
        if key != 'modules_to_not_convert':
            print(f"  {key}: {value}")
    
    obj = recommended.get('objectives', {})
    print(f"\n目標值:")
    print(f"  準確率變化: {obj.get('accuracy_change', 0)*100:+.2f}%")
    print(f"  GPU 峰值變化: {obj.get('gpu_peak_change', 0)*100:+.2f}%")
    print(f"  延遲變化: {obj.get('latency_change', 0)*100:+.2f}%")
    print(f"\n是否滿足目標: {'✓ 是' if recommended.get('satisfies_targets') else '✗ 否'}")

print("\n" + "="*70)


實驗摘要資訊

實驗名稱: gemma-2-2b-it-multiobjective-opt
時間戳記: 20260109_005815

總試驗數: 20
Pareto 解數量: 8
滿足目標解數: 2
優化器: optuna_multiobjective

--- 基準模型 ---
模型: google/gemma-2-2b-it
準確率: 59.00%
GPU 峰值: 5198 MB
平均延遲: 1621.29 ms

--- 優化目標 ---
準確率最小值: -10.0%
GPU 峰值最大值: -50.0%
延遲最大值: +150.0%

--- 推薦配置 ---
方法: gptq

配置詳情:
  method: gptq
  bits: 4
  group_size: 128
  calib_num: 256
  desc_act: True
  sym: False
  damp_percent: 0.001
  damp_auto_increment: 0.005
  act_group_aware: False
  static_groups: False
  true_sequential: False
  lm_head: False
  mse: 0.0
  rotation: None

目標值:
  準確率變化: -1.69%
  GPU 峰值變化: -53.69%
  延遲變化: +47.34%

是否滿足目標: ✓ 是



## 5. 視覺化 1：3D Pareto 前沿（含所有試驗）

顯示所有完成的試驗，並用顏色和形狀區分 Pareto 前沿和普通解。

In [9]:
# 收集所有完成的試驗（包括 Pareto 和非 Pareto）
if 'trials' in all_trials and all_trials['trials'] and 'solutions' in pareto:
    # 1. 獲取所有完成的試驗
    all_completed_trials = [t for t in all_trials['trials'] if t.get('status') == 'completed']
    
    # 2. 獲取 Pareto 前沿解（用於標記）
    pareto_trials = pareto['solutions']
    
    # 創建 Pareto 試驗的 ID 集合（用於標識）
    def trial_key(trial):
        """生成試驗的唯一鍵（用於識別是否在 Pareto 前沿）"""
        config = trial['config']
        return (config['method'], str(config))
    
    pareto_keys = {trial_key(t) for t in pareto_trials}
    
    # 3. 分類試驗
    pareto_data = []
    non_pareto_data = []
    
    for t in all_completed_trials:
        obj = t['objectives']
        config = t['config']
        
        # 跳過 None 值的試驗（失敗的試驗）
        if obj['accuracy_change'] is None or obj['gpu_peak_change'] is None or obj['latency_change'] is None:
            continue
        
        trial_info = {
            'acc': obj['accuracy_change'] * 100,
            'gpu': obj['gpu_peak_change'] * 100,
            'lat': obj['latency_change'] * 100,
            'method': config['method'],
            'config_short': format_config_short(config),
            'config_text': format_config(config),
            'satisfies_targets': t.get('satisfies_targets', False)
        }
        
        if trial_key(t) in pareto_keys:
            pareto_data.append(trial_info)
        else:
            non_pareto_data.append(trial_info)
    
    # 獲取目標約束
    targets = summary.get('targets', {})
    acc_min = targets.get('accuracy_min', -0.3) * 100  # -30%
    gpu_max = targets.get('gpu_peak_max', -0.3) * 100  # -30%
    lat_max = targets.get('latency_max', 1.0) * 100    # +100%
    
    # 建立圖表
    fig = go.Figure()
    
    # 計算數據範圍（使用所有試驗）
    all_acc = [t['acc'] for t in pareto_data + non_pareto_data]
    all_gpu = [t['gpu'] for t in pareto_data + non_pareto_data]
    all_lat = [t['lat'] for t in pareto_data + non_pareto_data]
    
    acc_range = max(all_acc) - min(all_acc)
    gpu_range = max(all_gpu) - min(all_gpu)
    lat_range = max(all_lat) - min(all_lat)
    
    # 目標區域的邊界
    acc_target_min = acc_min
    acc_target_max = max(all_acc) + acc_range * 0.1
    
    gpu_target_min = min(all_gpu) - gpu_range * 0.1
    gpu_target_max = gpu_max
    
    lat_target_min = min(all_lat) - lat_range * 0.1
    lat_target_max = lat_max
    
    # === 改用線框繪製目標區域（不會擋住點）===
    # 定義立方體的12條邊
    edges = [
        # 底面 (lat_min)
        ([acc_target_min, acc_target_max], [gpu_target_min, gpu_target_min], [lat_target_min, lat_target_min]),
        ([acc_target_max, acc_target_max], [gpu_target_min, gpu_target_max], [lat_target_min, lat_target_min]),
        ([acc_target_max, acc_target_min], [gpu_target_max, gpu_target_max], [lat_target_min, lat_target_min]),
        ([acc_target_min, acc_target_min], [gpu_target_max, gpu_target_min], [lat_target_min, lat_target_min]),
        
        # 頂面 (lat_max)
        ([acc_target_min, acc_target_max], [gpu_target_min, gpu_target_min], [lat_target_max, lat_target_max]),
        ([acc_target_max, acc_target_max], [gpu_target_min, gpu_target_max], [lat_target_max, lat_target_max]),
        ([acc_target_max, acc_target_min], [gpu_target_max, gpu_target_max], [lat_target_max, lat_target_max]),
        ([acc_target_min, acc_target_min], [gpu_target_max, gpu_target_min], [lat_target_max, lat_target_max]),
        
        # 垂直邊
        ([acc_target_min, acc_target_min], [gpu_target_min, gpu_target_min], [lat_target_min, lat_target_max]),
        ([acc_target_max, acc_target_max], [gpu_target_min, gpu_target_min], [lat_target_min, lat_target_max]),
        ([acc_target_max, acc_target_max], [gpu_target_max, gpu_target_max], [lat_target_min, lat_target_max]),
        ([acc_target_min, acc_target_min], [gpu_target_max, gpu_target_max], [lat_target_min, lat_target_max]),
    ]
    
    # 添加每條邊作為線條
    for i, (x_edge, y_edge, z_edge) in enumerate(edges):
        fig.add_trace(go.Scatter3d(
            x=x_edge,
            y=y_edge,
            z=z_edge,
            mode='lines',
            line=dict(color='lightgreen', width=3),
            showlegend=(i == 0),  # 只在第一條線顯示圖例
            name='目標區域',
            hoverinfo='skip'
        ))
    
    # 2. 添加非 Pareto 解（小點、半透明、灰色）
    if non_pareto_data:
        fig.add_trace(go.Scatter3d(
            x=[t['acc'] for t in non_pareto_data],
            y=[t['gpu'] for t in non_pareto_data],
            z=[t['lat'] for t in non_pareto_data],
            mode='markers',
            marker=dict(
                size=7,
                color='lightgray',
                opacity=0.5,
                line=dict(color='gray', width=1)
            ),
            text=[t['method'] for t in non_pareto_data],
            customdata=list(zip([t['config_short'] for t in non_pareto_data], 
                               [t['config_text'] for t in non_pareto_data])),
            hovertemplate='<b>非 Pareto 解</b><br>' +
                         '<b>Method: %{text}</b><br>' +
                         '<b>%{customdata[0]}</b><br><br>' +
                         '準確率變化: %{x:+.2f}%<br>' +
                         'GPU 峰值變化: %{y:+.2f}%<br>' +
                         '延遲變化: %{z:+.2f}%<br><br>' +
                         '<b>詳細配置：</b><br>%{customdata[1]}<extra></extra>',
            name='非 Pareto 解',
            showlegend=True
        ))
    
    # 3. 添加 Pareto 解（中等大小、明亮、圓形、根據滿足目標著色）
    if pareto_data:
        # 根據是否滿足目標設置顏色
        pareto_colors = ['green' if t['satisfies_targets'] else 'blue' for t in pareto_data]
        
        fig.add_trace(go.Scatter3d(
            x=[t['acc'] for t in pareto_data],
            y=[t['gpu'] for t in pareto_data],
            z=[t['lat'] for t in pareto_data],
            mode='markers+text',
            marker=dict(
                size=10,  # 減小到 10
                color=pareto_colors,
                opacity=0.95,
                line=dict(color='white', width=2)
            ),
            text=[t['method'] for t in pareto_data],
            textposition='top center',
            textfont=dict(size=9),  # 移除粗體，減小字體
            customdata=list(zip([t['config_short'] for t in pareto_data], 
                               [t['config_text'] for t in pareto_data])),
            hovertemplate='<b>⭐ Pareto 最優解</b><br>' +
                         '<b>Method: %{text}</b><br>' +
                         '<b>%{customdata[0]}</b><br><br>' +
                         '準確率變化: %{x:+.2f}%<br>' +
                         'GPU 峰值變化: %{y:+.2f}%<br>' +
                         '延遲變化: %{z:+.2f}%<br><br>' +
                         '<b>詳細配置：</b><br>%{customdata[1]}<extra></extra>',
            name='Pareto 前沿',
            showlegend=True
        ))
    
    fig.update_layout(
        title=dict(
            text='3D Pareto 前沿 - 所有試驗視覺化<br><sub>灰色小點 = 非 Pareto 解 | 藍色圓點 = Pareto 解（未滿足目標）| 綠色圓點 = Pareto 解（滿足目標）| 綠色線框 = 目標範圍<br>將滑鼠移到點上可查看配置詳情</sub>',
            x=0.5,
            xanchor='center'
        ),
        scene=dict(
            xaxis_title='準確率變化 (%)<br>[↑ 正 = 上升]',
            yaxis_title='GPU 峰值變化 (%)<br>[↓ 負 = 減少]',
            zaxis_title='延遲變化 (%)<br>[↓ 負 = 加速]',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.3)
            )
        ),
        width=1200,
        height=800,
        showlegend=True,
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor='rgba(255,255,255,0.8)'
        ),
        hovermode='closest'
    )
    
    # 統計
    print(f"\n=== 3D 視覺化統計 ===")
    print(f"總試驗數（完成）: {len(pareto_data) + len(non_pareto_data)}")
    print(f"  Pareto 前沿: {len(pareto_data)} 個")
    print(f"  非 Pareto 解: {len(non_pareto_data)} 個")
    print(f"  滿足目標的 Pareto 解: {sum(1 for t in pareto_data if t['satisfies_targets'])} 個")
    print(f"\n目標約束：")
    print(f"  準確率變化 ≥ {acc_min:+.1f}%")
    print(f"  GPU 峰值變化 ≤ {gpu_max:+.1f}%")
    print(f"  延遲變化 ≤ {lat_max:+.1f}%")
    
    fig.show()
else:
    print("未找到試驗資料或 Pareto 解資料")


=== 3D 視覺化統計 ===
總試驗數（完成）: 16
  Pareto 前沿: 9 個
  非 Pareto 解: 7 個
  滿足目標的 Pareto 解: 3 個

目標約束：
  準確率變化 ≥ -10.0%
  GPU 峰值變化 ≤ -50.0%
  延遲變化 ≤ +150.0%


## 6. 視覺化 2：Pareto 解詳細表格

In [10]:
# 使用 'solutions' 而非 'pareto_solutions'
if 'solutions' in pareto and pareto['solutions']:
    pareto_trials = pareto['solutions']
    
    # 建立表格資料
    table_data = []
    for i, t in enumerate(pareto_trials, 1):
        obj = t['objectives']
        config = t['config']
        table_data.append({
            '序號': i,
            '方法': config['method'],
            '位元數': config.get('bits', config.get('w_bit', 'N/A')),
            '群組大小': config.get('group_size', config.get('q_group_size', 'N/A')),
            '準確率變化': f"{obj['accuracy_change']*100:+.2f}%",
            'GPU變化': f"{obj['gpu_peak_change']*100:+.2f}%",
            '延遲變化': f"{obj['latency_change']*100:+.2f}%",
            '滿足目標': '✓' if t.get('satisfies_targets') else '✗'
        })
    
    df_pareto = pd.DataFrame(table_data)
    
    print("\n=== Pareto 前沿解詳細資訊 ===")
    display(df_pareto)
    
    # 匯出為 CSV
    # output_csv = RESULTS_DIR / 'optimization' / SELECTED_EXPERIMENT / 'pareto_solutions_table.csv'
    # df_pareto.to_csv(output_csv, index=False, encoding='utf-8-sig')
    # print(f"\n表格已匯出至: {output_csv}")


=== Pareto 前沿解詳細資訊 ===


,序號,方法,位元數,群組大小,準確率變化,GPU變化,延遲變化,滿足目標
0,1,gptq,8,-1,+1.69%,-36.49%,+41.35%,✗
1,2,gptq,3,64,-6.78%,-56.91%,+189.81%,✗
2,3,gptq,4,128,-5.08%,-53.73%,+49.74%,✓
3,4,gptq,8,-1,+1.69%,-36.50%,+42.56%,✗
4,5,gptq,2,16,-94.92%,-59.02%,+114.12%,✗
5,6,gptq,4,128,-1.69%,-53.69%,+47.34%,✓
6,7,gptq,4,128,-10.17%,-53.63%,+8.57%,✗
7,8,gptq,4,128,-10.17%,-53.72%,+8.80%,✗


## 6.5 All Trials 詳細表格

顯示所有試驗的詳細資訊，包括成功、失敗、和被剪枝的試驗。

In [24]:
if 'trials' in all_trials and all_trials['trials']:
    trials = all_trials['trials']

    # 建立表格資料
    table_data = []
    for i, t in enumerate(trials, 1):
        config = t['config']
        obj = t.get('objectives', {})

        # 處理 None 值
        acc_change = obj.get('accuracy_change')
        gpu_change = obj.get('gpu_peak_change')
        lat_change = obj.get('latency_change')

        # 格式化目標值（處理 None）
        acc_str = f"{acc_change*100:+.2f}%" if acc_change is not None else "N/A"
        gpu_str = f"{gpu_change*100:+.2f}%" if gpu_change is not None else "N/A"
        lat_str = f"{lat_change*100:+.2f}%" if lat_change is not None else "N/A"
        vio_str = f"{t.get('violation_score', 0.0):.4f}" if t.get('violation_score') is not None else "N/A"

        # 提取關鍵配置參數
        method = config.get('method', 'unknown')
        if method == 'gptq':
            bits = config.get('bits', 'N/A')
            group_size = config.get('group_size', 'N/A')
        elif method == 'awq':
            bits = config.get('w_bit', 'N/A')
            group_size = config.get('q_group_size', 'N/A')
        elif method == 'bnb':
            bits = config.get('bits', 'N/A')
            group_size = 'N/A'
        else:
            bits = 'N/A'
            group_size = 'N/A'

        # 已經明確使用過的 config keys
        used_keys = set()

        used_keys.add('method')

        if method == 'gptq':
            used_keys.update(['bits', 'group_size'])
        elif method == 'awq':
            used_keys.update(['w_bit', 'q_group_size'])
        elif method == 'bnb':
            used_keys.update(['bits'])
        
        # 收集其餘未使用的 config
        detail_items = []

        for k, v in config.items():
            if k in used_keys:
                continue

            # 讓輸出更穩定、可讀
            if isinstance(v, float):
                v_str = f"{v:.4g}"
            elif isinstance(v, (list, dict)):
                v_str = str(v)
            else:
                v_str = str(v)

            detail_items.append(f"{k}={v_str}")

        # 最終 detail_config
        detail_config = ", ".join(detail_items) if detail_items else ""

        # 狀態標記
        status = t.get('status', 'unknown')
        status_icon = {
            'completed': '✓',
            'failed': '✗',
            'pruned': '⊗'
        }.get(status, '?')

        table_data.append({
            'Trial': i,
            '狀態': f"{status_icon} {status}",
            '方法': method,
            '位元數': bits,
            '群組大小': group_size,
            '準確率變化': acc_str,
            'GPU變化': gpu_str,
            '延遲變化': lat_str,
            'Violation Score': vio_str,
            '滿足約束': '✓' if t.get('satisfies_constraints') else '✗',
            '滿足目標': '✓' if t.get('satisfies_targets') else '✗',
            '錯誤': t.get('error', '')[:50] + '...' if t.get('error') and len(t.get('error', '')) > 50 else t.get('error', ''),
            '參數': detail_config
        })

    df_all_trials = pd.DataFrame(table_data)

    # 統計
    completed = len([t for t in trials if t.get('status') == 'completed'])
    failed = len([t for t in trials if t.get('status') == 'failed'])
    pruned = len([t for t in trials if t.get('status') == 'pruned'])
    satisfies_constraints = len([t for t in trials if t.get('satisfies_constraints')])
    satisfies_targets = len([t for t in trials if t.get('satisfies_targets')])

    print("\n=== All Trials 詳細資訊 ===")
    print(f"總試驗數: {len(trials)}")
    print(f"  ✓ 完成: {completed}")
    print(f"  ✗ 失敗: {failed}")
    print(f"  ⊗ 被剪枝: {pruned}")
    print(f"\n滿足約束的試驗: {satisfies_constraints} ({satisfies_constraints/len(trials)*100:.1f}%)")
    print(f"滿足目標的試驗: {satisfies_targets} ({satisfies_targets/len(trials)*100:.1f}%)")
    print()

    # 顯示表格
    display(df_all_trials.drop('參數', axis=1).style.hide(axis="index"))

    # 可選：匯出為 CSV
    # output_csv = RESULTS_DIR / 'optimization' / SELECTED_EXPERIMENT / 'all_trials_table.csv'
    # df_all_trials.to_csv(output_csv, index=False, encoding='utf-8-sig')
    # print(f"\n表格已匯出至: {output_csv}")
else:
    print("未找到試驗資料")


=== All Trials 詳細資訊 ===
總試驗數: 20
  ✓ 完成: 16
  ✗ 失敗: 4
  ⊗ 被剪枝: 0

滿足約束的試驗: 16 (80.0%)
滿足目標的試驗: 3 (15.0%)



Trial,狀態,方法,位元數,群組大小,準確率變化,GPU變化,延遲變化,Violation Score,滿足約束,滿足目標,錯誤
1,✓ completed,gptq,8,-1,+1.69%,-36.49%,+41.35%,0.1351,✓,✗,
2,✗ failed,awq,4,128,N/A,N/A,N/A,N/A,✗,✗,'Catcher' object has no attribute 'attention_type'
3,✗ failed,awq,4,16,N/A,N/A,N/A,N/A,✗,✗,'Catcher' object has no attribute 'attention_type'
4,✓ completed,gptq,3,64,-6.78%,-56.91%,+189.81%,0.3981,✓,✗,
5,✗ failed,awq,4,16,N/A,N/A,N/A,N/A,✗,✗,'Catcher' object has no attribute 'attention_type'
6,✓ completed,gptq,4,128,-5.08%,-53.66%,+49.55%,0.0000,✓,✓,
7,✗ failed,awq,4,32,N/A,N/A,N/A,N/A,✗,✗,'Catcher' object has no attribute 'attention_type'
8,✓ completed,bnb,4,N/A,-18.64%,-19.05%,+41.08%,0.4824,✓,✗,
9,✓ completed,gptq,4,128,-5.08%,-53.73%,+49.74%,0.0000,✓,✓,
10,✓ completed,bnb,8,N/A,-13.56%,-36.51%,+241.56%,1.1216,✓,✗,


In [34]:
df_style = df_all_trials[df_all_trials['狀態'].str.contains('failed') == False].copy()

def percent_to_float(x):
    if isinstance(x, str) and x.endswith('%'):
        try:
            return float(x.replace('%', '')) / 100
        except:
            return np.nan
    return np.nan

df_style['_acc'] = df_style['準確率變化'].apply(percent_to_float)
df_style['_gpu'] = df_style['GPU變化'].apply(percent_to_float)
df_style['_lat'] = df_style['延遲變化'].apply(percent_to_float)
df_style['_vio'] = pd.to_numeric(df_style['Violation Score'], errors='coerce')

styled_df = (
    df_style
    .style
    # 準確率：高 → 綠
    .background_gradient(
        subset=['準確率變化'],
        cmap='Greens',
        gmap=df_style['_acc']
    )
    # GPU：低 → 綠
    .background_gradient(
        subset=['GPU變化'],
        cmap='RdYlGn_r',
        gmap=df_style['_gpu']
    )
    # 延遲：低 → 綠
    .background_gradient(
        subset=['延遲變化'],
        cmap='RdYlGn_r',
        gmap=df_style['_lat']
    )
    # Violation Score：低 → 綠
    .background_gradient(
        subset=['Violation Score'],
        cmap='RdYlGn_r',
        gmap=df_style['_vio']
    )
)

display_columns = [
    col for col in df_style.columns
    if not col.startswith('_')
    and col not in ['錯誤', '狀態', '參數']
]

# 先決定要顯示的 DataFrame
df_display = df_style[display_columns].copy()

# tooltip 建立
tooltip_df = pd.DataFrame("", index=df_display.index, columns=df_display.columns)
tooltip_df['方法'] = df_style.loc[df_display.index, '參數'].fillna("").apply(
    lambda x: f"Config:\n{x}" if x else ""
)

# 建立 Styler
styled_df = df_display.style

# 加上顏色
for col, gmap_col in zip(
    ['準確率變化','GPU變化','延遲變化','Violation Score'],
    ['_acc','_gpu','_lat','_vio']
):
    cmap = 'Greens' if col == '準確率變化' else 'RdYlGn_r'
    styled_df = styled_df.background_gradient(
        subset=[col],
        cmap=cmap,
        gmap=df_style.loc[df_display.index, gmap_col]
    )

# 加上 tooltip
styled_df = styled_df.set_tooltips(tooltip_df).hide(axis='index')

display(styled_df)



Trial,方法,位元數,群組大小,準確率變化,GPU變化,延遲變化,Violation Score,滿足約束,滿足目標
1,gptq,8,-1,+1.69%,-36.49%,+41.35%,0.1351,✓,✗
4,gptq,3,64,-6.78%,-56.91%,+189.81%,0.3981,✓,✗
6,gptq,4,128,-5.08%,-53.66%,+49.55%,0.0000,✓,✓
8,bnb,4,N/A,-18.64%,-19.05%,+41.08%,0.4824,✓,✗
9,gptq,4,128,-5.08%,-53.73%,+49.74%,0.0000,✓,✓
10,bnb,8,N/A,-13.56%,-36.51%,+241.56%,1.1216,✓,✗
11,gptq,4,32,-16.95%,-51.61%,+35.02%,0.1390,✓,✗
12,gptq,8,-1,+1.69%,-36.50%,+42.56%,0.1350,✓,✗
13,gptq,8,-1,-11.86%,-36.50%,+42.43%,0.1723,✓,✗
14,gptq,2,16,-94.92%,-59.02%,+114.12%,1.6983,✓,✗


## 7. 視覺化 3：量化方法比較

In [10]:
if 'trials' in all_trials:
    trials = all_trials['trials']
    
    # 按方法分組並計算統計（過濾掉有 None 值的試驗）
    method_groups = {}
    for t in trials:
        if 'objectives' in t and all(k in t['objectives'] for k in ['accuracy_change', 'gpu_peak_change', 'latency_change']):
            # 檢查是否有 None 值
            if all(t['objectives'][k] is not None for k in ['accuracy_change', 'gpu_peak_change', 'latency_change']):
                method = t['config']['method']
                if method not in method_groups:
                    method_groups[method] = []
                method_groups[method].append(t)
    
    # 計算每個方法的統計資料
    method_stats = []
    for method, trials_list in method_groups.items():
        avg_acc = sum(t['objectives']['accuracy_change'] for t in trials_list) / len(trials_list) * 100
        avg_gpu = sum(t['objectives']['gpu_peak_change'] for t in trials_list) / len(trials_list) * 100
        avg_lat = sum(t['objectives']['latency_change'] for t in trials_list) / len(trials_list) * 100
        
        # 計算標準差
        std_acc = pd.Series([t['objectives']['accuracy_change']*100 for t in trials_list]).std()
        std_gpu = pd.Series([t['objectives']['gpu_peak_change']*100 for t in trials_list]).std()
        std_lat = pd.Series([t['objectives']['latency_change']*100 for t in trials_list]).std()
        
        # 處理 NaN 標準差（當只有一個樣本時）
        std_acc = 0 if pd.isna(std_acc) else std_acc
        std_gpu = 0 if pd.isna(std_gpu) else std_gpu
        std_lat = 0 if pd.isna(std_lat) else std_lat
        
        method_stats.append({
            'method': method,
            'avg_acc': avg_acc,
            'avg_gpu': avg_gpu,
            'avg_lat': avg_lat,
            'std_acc': std_acc,
            'std_gpu': std_gpu,
            'std_lat': std_lat,
            'count': len(trials_list)
        })
    
    if method_stats:
        df_methods = pd.DataFrame(method_stats).sort_values('avg_acc', ascending=False)
        
        # 建立比較圖
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                '平均準確率變化',
                '平均 GPU 峰值變化',
                '平均延遲變化',
                '試驗次數分佈'
            )
        )
        
        # 1. 準確率（帶誤差棒）
        fig.add_trace(
            go.Bar(
                x=df_methods['method'],
                y=df_methods['avg_acc'],
                error_y=dict(type='data', array=df_methods['std_acc']),
                marker_color='lightblue',
                text=[f"{v:+.1f}%" for v in df_methods['avg_acc']],
                textposition='outside',
                name='準確率'
            ),
            row=1, col=1
        )
        
        # 2. GPU（帶誤差棒）
        fig.add_trace(
            go.Bar(
                x=df_methods['method'],
                y=df_methods['avg_gpu'],
                error_y=dict(type='data', array=df_methods['std_gpu']),
                marker_color='lightgreen',
                text=[f"{v:+.1f}%" for v in df_methods['avg_gpu']],
                textposition='outside',
                name='GPU'
            ),
            row=1, col=2
        )
        
        # 3. 延遲（帶誤差棒）
        fig.add_trace(
            go.Bar(
                x=df_methods['method'],
                y=df_methods['avg_lat'],
                error_y=dict(type='data', array=df_methods['std_lat']),
                marker_color='lightcoral',
                text=[f"{v:+.1f}%" for v in df_methods['avg_lat']],
                textposition='outside',
                name='延遲'
            ),
            row=2, col=1
        )
        
        # 4. 試驗次數
        fig.add_trace(
            go.Bar(
                x=df_methods['method'],
                y=df_methods['count'],
                marker_color='lightyellow',
                text=df_methods['count'],
                textposition='outside',
                name='次數'
            ),
            row=2, col=2
        )
        
        # 添加參考線（0%）
        for row, col in [(1, 1), (1, 2), (2, 1)]:
            fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=row, col=col)
        
        fig.update_yaxes(title_text="變化 (%)", row=1, col=1)
        fig.update_yaxes(title_text="變化 (%)", row=1, col=2)
        fig.update_yaxes(title_text="變化 (%)", row=2, col=1)
        fig.update_yaxes(title_text="試驗次數", row=2, col=2)
        
        fig.update_layout(
            title_text="量化方法平均表現比較（含標準差）",
            height=700,
            showlegend=False
        )
        
        fig.show()
        
        # 顯示統計表
        print("\n=== 方法統計 ===")
        df_display = df_methods.copy()
        df_display['avg_acc'] = df_display.apply(lambda x: f"{x['avg_acc']:+.2f}% ± {x['std_acc']:.2f}%", axis=1)
        df_display['avg_gpu'] = df_display.apply(lambda x: f"{x['avg_gpu']:+.2f}% ± {x['std_gpu']:.2f}%", axis=1)
        df_display['avg_lat'] = df_display.apply(lambda x: f"{x['avg_lat']:+.2f}% ± {x['std_lat']:.2f}%", axis=1)
        df_display = df_display[['method', 'avg_acc', 'avg_gpu', 'avg_lat', 'count']]
        df_display.columns = ['方法', '平均準確率變化', '平均GPU變化', '平均延遲變化', '試驗次數']
        display(df_display)
    else:
        print("沒有有效的試驗資料可供分析")
else:
    print("未找到試驗資料")


=== 方法統計 ===


,方法,平均準確率變化,平均GPU變化,平均延遲變化,試驗次數
0,gptq,-14.69% ± 26.02%,-49.76% ± 8.25%,+55.40% ± 49.86%,12
1,bnb,-15.68% ± 3.49%,-27.77% ± 10.07%,+141.60% ± 116.16%,4


## 8. 視覺化 4: 平行座標圖 (Parallel Coordinates)

In [28]:
# 從 all_trials.json 讀取完整資料（包含 violation_score）
# 然後只選擇 Pareto 解來顯示

if 'solutions' in pareto and pareto['solutions']:
    # 建立 Pareto 解的識別函數
    def trial_key(t):
        config = t['config']
        return (config['method'], 
                config.get('bits') or config.get('w_bit'),
                config.get('group_size') or config.get('q_group_size', -1))
    
    # 建立 Pareto 解的 key 集合
    pareto_keys = {trial_key(t) for t in pareto['solutions']}
    
    # 從 all_trials 中找出對應的完整資料（包含 violation_score）
    pareto_with_vs = []
    for t in all_trials['trials']:
        if t.get('status') == 'completed' and trial_key(t) in pareto_keys:
            pareto_with_vs.append(t)
    
    # 準備資料 - DataFrame 的所有列都會自動顯示在 hover 中
    data = []
    
    for idx, t in enumerate(pareto_with_vs, 1):
        config = t['config']
        objectives = t['objectives']
        
        row = {
            'Trial': idx,
            'Method': config['method'],
            'Accuracy (%)': round(objectives['accuracy_change'] * 100, 2),
            'GPU Saving (%)': round(-objectives['gpu_peak_change'] * 100, 2),
            'Latency Change (%)': round(-objectives['latency_change'] * 100, 2),
            'Violation Score': round(t.get('violation_score', 0.0), 4)
        }
        
        # 添加方法特定的參數（這些會在 hover 中顯示）
        if config['method'] == 'gptq':
            row['Bits'] = config.get('bits', 'N/A')
            row['Group Size'] = config.get('group_size', 'N/A')
            row['Desc Act'] = config.get('desc_act', 'N/A')
            row['Symmetric'] = config.get('sym', 'N/A')
            row['Damp %'] = config.get('damp_percent', 'N/A')
        elif config['method'] == 'awq':
            row['Bits'] = config.get('w_bit', 'N/A')
            row['Group Size'] = config.get('q_group_size', 'N/A')
            row['Zero Point'] = config.get('zero_point', 'N/A')
        elif config['method'] == 'bnb':
            row['Bits'] = config.get('bits', 'N/A')
            row['Quant Type'] = config.get('bnb_4bit_quant_type', 'N/A')
        
        data.append(row)
    
    # 轉換為 DataFrame
    import pandas as pd
    df = pd.DataFrame(data)
    
    print(f"\nPareto 解的 violation_score 分佈：")
    print(f"  最小值（最接近目標）: {df['Violation Score'].min():.4f}")
    print(f"  最大值（最遠離目標）: {df['Violation Score'].max():.4f}")
    print(f"  平均值: {df['Violation Score'].mean():.4f}")
    
    # 目標參考值
    targets = {
        'accuracy_min': -10,
        'gpu_peak_max': 50,
        'latency_max': -150
    }
    
    # 建立平行座標圖，並使用 constraintrange 標示達標區域
    import plotly.graph_objects as go
    
    fig = go.Figure(data=
        go.Parcoords(
            line=dict(
                color=df['Violation Score'],
                colorscale=[
                    [0.0, 'rgb(0, 128, 0)'],      # 深綠（低違反，好）
                    [0.5, 'rgb(255, 140, 0)'],    # 深橙（中等）
                    [1.0, 'rgb(220, 20, 60)']     # 深紅（高違反，差）
                ],
                showscale=True,
                cmin=0,
                cmax=df['Violation Score'].max(),
                colorbar=dict(
                    title="Violation Score<br>(越低越好)",
                    len=0.7
                )
            ),
            dimensions=[
                dict(
                    range=[df['Accuracy (%)'].min() - 5, df['Accuracy (%)'].max() + 5],
                    constraintrange=[targets['accuracy_min'], df['Accuracy (%)'].max() + 5],  # 達標區域
                    label='準確率變化 (%)',
                    values=df['Accuracy (%)']
                ),
                dict(
                    range=[df['GPU Saving (%)'].min() - 5, df['GPU Saving (%)'].max() + 5],
                    constraintrange=[targets['gpu_peak_max'], df['GPU Saving (%)'].max() + 5],  # 達標區域
                    label='GPU 峰值節省 (%) [↑ = 節省更多]',
                    values=df['GPU Saving (%)']
                ),
                dict(
                    range=[df['Latency Change (%)'].min() - 5, df['Latency Change (%)'].max() + 5],
                    constraintrange=[targets['latency_max'], df['Latency Change (%)'].max() + 5],  # 達標區域
                    label='延遲變化 (%) [↑ = 加速更多]',
                    values=df['Latency Change (%)']
                )
            ]
        )
    )
    
    # 更新佈局
    fig.update_layout(
        title='Pareto 解 - 平行座標圖（依違反分數著色，綠色區域為達標範圍）',
        height=650,
        width=1200
    )
    
    # === 添加目標線標註（使用 annotation 在軸旁邊）===
    
    # 1. 準確率目標標註
    fig.add_annotation(
        x=0.05, y=0.02,
        xref='paper', yref='paper',
        text=f"🎯 準確率目標: ≥ {targets['accuracy_min']}%",
        showarrow=False,
        font=dict(size=10, color='green', family='Arial Black'),
        xanchor='left',
        bgcolor='rgba(144, 238, 144, 0.3)',
        bordercolor='green',
        borderwidth=2,
        borderpad=4
    )
    
    # 2. GPU 節省目標標註
    fig.add_annotation(
        x=0.4, y=0.02,
        xref='paper', yref='paper',
        text=f"🎯 GPU 目標: ≥ {targets['gpu_peak_max']}%",
        showarrow=False,
        font=dict(size=10, color='blue', family='Arial Black'),
        xanchor='center',
        bgcolor='rgba(173, 216, 230, 0.3)',
        bordercolor='blue',
        borderwidth=2,
        borderpad=4
    )
    
    # 3. 延遲變化目標標註
    fig.add_annotation(
        x=0.95, y=0.02,
        xref='paper', yref='paper',
        text=f"🎯 延遲目標: ≥ {targets['latency_max']}%",
        showarrow=False,
        font=dict(size=10, color='purple', family='Arial Black'),
        xanchor='right',
        bgcolor='rgba(221, 160, 221, 0.3)',
        bordercolor='purple',
        borderwidth=2,
        borderpad=4
    )
    
    fig.show()
    
    print(f"\n📊 平行座標圖顯示了 {len(pareto_with_vs)} 個 Pareto 解")
    
    # 顯示 DataFrame 預覽
    print("\n📋 資料預覽（hover 會顯示所有這些欄位）：")
    print(df.to_string())
else:
    print("未找到 Pareto 解資料")


Pareto 解的 violation_score 分佈：
  最小值（最接近目標）: 0.0000
  最大值（最遠離目標）: 1.6983
  平均值: 0.2546



📊 平行座標圖顯示了 10 個 Pareto 解

📋 資料預覽（hover 會顯示所有這些欄位）：
   Trial Method  Accuracy (%)  GPU Saving (%)  Latency Change (%)  Violation Score  Bits  Group Size  Desc Act  Symmetric  Damp %
0      1   gptq          1.69           36.49              -41.35           0.1351     8          -1      True      False   0.005
1      2   gptq         -6.78           56.91             -189.81           0.3981     3          64     False       True   0.005
2      3   gptq         -5.08           53.66              -49.55           0.0000     4         128      True      False   0.030
3      4   gptq         -5.08           53.73              -49.74           0.0000     4         128      True      False   0.030
4      5   gptq          1.69           36.50              -42.56           0.1350     8          -1      True      False   0.005
5      6   gptq        -11.86           36.50              -42.43           0.1723     8          -1      True      False   0.050
6      7   gptq        -94.92         

## 9. 視覺化 5：權衡散點圖矩陣 (Trade-off Matrix)

2x2 矩陣顯示三個目標之間的兩兩權衡關係

In [ ]:
# 使用 'solutions' 而非 'pareto_solutions'
if 'solutions' in pareto and pareto['solutions']:
    pareto_trials = pareto['solutions']
    
    # 提取資料
    acc_changes = [t['objectives']['accuracy_change'] * 100 for t in pareto_trials]
    gpu_changes = [t['objectives']['gpu_peak_change'] * 100 for t in pareto_trials]
    latency_changes = [t['objectives']['latency_change'] * 100 for t in pareto_trials]
    methods = [t['config']['method'] for t in pareto_trials]
    
    # 格式化配置信息
    config_short = [format_config_short(t['config']) for t in pareto_trials]
    config_text = [format_config(t['config']) for t in pareto_trials]
    
    # 建立子圖（2x2 矩陣）
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Accuracy vs GPU', 'Accuracy vs Latency',
                      'GPU vs Latency', 'Method Distribution')
    )
    
    # 1. Accuracy vs GPU
    fig.add_trace(
        go.Scatter(
            x=acc_changes, y=gpu_changes, mode='markers',
            marker=dict(size=10, color='blue', opacity=0.6),
            text=config_short,
            customdata=config_text,
            hovertemplate='<b>%{text}</b><br>' +
                         'Accuracy: %{x:+.2f}%<br>' +
                         'GPU: %{y:+.2f}%<br><br>' +
                         '<b>配置：</b><br>%{customdata}<extra></extra>',
            name='', showlegend=False
        ),
        row=1, col=1
    )
    fig.update_xaxes(title_text="準確率變化 (%)", row=1, col=1)
    fig.update_yaxes(title_text="GPU 峰值變化 (%)", row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=1)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=1)
    
    # 2. Accuracy vs Latency
    fig.add_trace(
        go.Scatter(
            x=acc_changes, y=latency_changes, mode='markers',
            marker=dict(size=10, color='green', opacity=0.6),
            text=config_short,
            customdata=config_text,
            hovertemplate='<b>%{text}</b><br>' +
                         'Accuracy: %{x:+.2f}%<br>' +
                         'Latency: %{y:+.2f}%<br><br>' +
                         '<b>配置：</b><br>%{customdata}<extra></extra>',
            name='', showlegend=False
        ),
        row=1, col=2
    )
    fig.update_xaxes(title_text="準確率變化 (%)", row=1, col=2)
    fig.update_yaxes(title_text="延遲變化 (%)", row=1, col=2)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=2)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.3, row=1, col=2)
    
    # 3. GPU vs Latency
    fig.add_trace(
        go.Scatter(
            x=gpu_changes, y=latency_changes, mode='markers',
            marker=dict(size=10, color='red', opacity=0.6),
            text=config_short,
            customdata=config_text,
            hovertemplate='<b>%{text}</b><br>' +
                         'GPU: %{x:+.2f}%<br>' +
                         'Latency: %{y:+.2f}%<br><br>' +
                         '<b>配置：</b><br>%{customdata}<extra></extra>',
            name='', showlegend=False
        ),
        row=2, col=1
    )
    fig.update_xaxes(title_text="GPU 峰值變化 (%)", row=2, col=1)
    fig.update_yaxes(title_text="延遲變化 (%)", row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.3, row=2, col=1)
    fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.3, row=2, col=1)
    
    # 4. Method distribution
    from collections import Counter
    method_counts = Counter(methods)
    
    fig.add_trace(
        go.Bar(x=list(method_counts.keys()), y=list(method_counts.values()),
              marker_color='lightblue',
              text=list(method_counts.values()),
              textposition='outside',
              name='', showlegend=False),
        row=2, col=2
    )
    fig.update_xaxes(title_text="Method", row=2, col=2)
    fig.update_yaxes(title_text="Count", row=2, col=2)
    
    fig.update_layout(
        height=800, 
        width=1200, 
        title_text="Trade-off Analysis Matrix<br><sub>將滑鼠移到點上可查看配置詳情</sub>"
    )
    
    fig.show()
    
    print("\n提示：將滑鼠移到散點上可以查看詳細的量化配置參數")
else:
    print("未找到 Pareto 解資料")


提示：將滑鼠移到散點上可以查看詳細的量化配置參數


## 11. 視覺化 6：每個資料集的詳細結果

In [ ]:
# 顯示推薦配置在各資料集的表現
recommended = summary.get('recommended', {})

if recommended and 'per_dataset' in recommended:
    per_dataset = recommended['per_dataset']
    
    # 準備資料
    dataset_results = []
    for dataset, metrics in per_dataset.items():
        dataset_results.append({
            '資料集': dataset,
            '準確率': f"{metrics.get('accuracy', 0)*100:.2f}%",
            'GPU峰值(MB)': f"{metrics.get('gpu_peak_mb', 0):.0f}",
            '樣本數': metrics.get('num_samples', 'N/A'),
            '正確數': metrics.get('correct', 'N/A'),
            '平均延遲(ms)': f"{metrics.get('avg_latency_ms', 0):.2f}",
            '吞吐量': f"{metrics.get('throughput_tokens_per_sec', 0):.0f}"
        })
    
    df_datasets = pd.DataFrame(dataset_results)
    
    print("\n=== 推薦配置在各資料集的表現 ===")
    display(df_datasets)
    
    # 視覺化
    accuracies = [float(r['準確率'].replace('%', '')) for r in dataset_results]
    datasets = [r['資料集'] for r in dataset_results]
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=datasets,
        y=accuracies,
        text=[f"{a:.2f}%" for a in accuracies],
        textposition='outside',
        marker_color='lightseagreen'
    ))
    
    fig.update_layout(
        title='推薦配置在不同資料集的準確率',
        xaxis_title='資料集',
        yaxis_title='準確率 (%)',
        height=400
    )
    
    fig.show()
else:
    print("\n未找到各資料集的詳細結果")


=== 推薦配置在各資料集的表現 ===


,資料集,準確率,GPU峰值(MB),樣本數,正確數,平均延遲(ms),吞吐量
0,gsm8k,56.00%,2407,50,28,2637.34,459
1,truthfulqa,60.00%,2243,50,30,2140.36,104


## 12. 匯出完整報告

In [ ]:
# 匯出所有圖表和表格的摘要
from datetime import datetime

report_dir = RESULTS_DIR / 'optimization' / SELECTED_EXPERIMENT
report_file = report_dir / f"visualization_report.txt"

with open(report_file, 'w', encoding='utf-8') as f:
    f.write("="*70 + "\n")
    f.write("優化實驗視覺化報告\n")
    f.write("="*70 + "\n\n")
    
    # 1. 基本信息
    f.write("【實驗基本信息】\n")
    f.write(f"實驗名稱: {summary.get('experiment_name', 'N/A')}\n")
    f.write(f"時間戳記: {summary.get('timestamp', 'N/A')}\n")
    f.write(f"報告生成時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("\n")
    
    # 2. 優化器信息
    opt_info = summary.get('optimization', {})
    f.write("【優化器配置】\n")
    f.write(f"優化器類型: {opt_info.get('optimizer', 'N/A')}\n")
    f.write(f"總試驗數: {opt_info.get('total_trials', 'N/A')}\n")
    f.write(f"Pareto 解數量: {opt_info.get('pareto_solutions', 'N/A')}\n")
    f.write(f"滿足目標解數: {opt_info.get('satisfying_solutions', 'N/A')}\n")
    f.write("\n")
    
    # 3. 優化目標
    targets = summary.get('targets', {})
    f.write("【優化目標】\n")
    f.write(f"目標1 - 準確率變化: ≥ {targets.get('accuracy_min', 0)*100:+.1f}%（相對於基準模型）\n")
    f.write(f"目標2 - GPU 峰值變化: ≤ {targets.get('gpu_peak_max', 0)*100:+.1f}%（相對於基準模型）\n")
    f.write(f"目標3 - 延遲變化: ≤ {targets.get('latency_max', 0)*100:+.1f}%（相對於基準模型）\n")
    f.write("\n")
    
    # 4. 基準模型表現
    baseline = summary.get('baseline', {})
    f.write("【基準模型表現】\n")
    f.write(f"模型: {baseline.get('model', 'N/A')}\n")
    f.write(f"準確率: {baseline.get('accuracy', 0)*100:.2f}%\n")
    f.write(f"GPU 峰值: {baseline.get('gpu_peak_mb', 0):.0f} MB\n")
    f.write(f"平均延遲: {baseline.get('avg_latency_ms', 0):.2f} ms\n")
    f.write("\n")
    
    
    
    # 5. Pareto 前沿概覽
    if 'solutions' in pareto and pareto['solutions']:
        f.write("【Pareto 前沿概覽】\n")
        f.write(f"Pareto 前沿包含 {len(pareto['solutions'])} 個非支配解\n")
        f.write(f"其中 {sum(1 for s in pareto['solutions'] if s.get('satisfies_targets'))} 個滿足所有目標約束\n")
        f.write("\n")
        
        f.write("Pareto 前沿解列表:\n")
        for i, sol in enumerate(pareto['solutions'], 1):
            config = sol['config']
            obj = sol['objectives']
            method = config.get('method', 'unknown')
            
            # 簡短配置描述
            if method == 'gptq':
                cfg_str = f"bits={config.get('bits', '?')}, gs={config.get('group_size', '?')}"
            elif method == 'awq':
                cfg_str = f"w_bit={config.get('w_bit', '?')}, gs={config.get('q_group_size', '?')}"
            elif method == 'bnb':
                cfg_str = f"bits={config.get('bits', '?')}, type={config.get('bnb_4bit_quant_type', '?')}"
            else:
                cfg_str = "N/A"
            
            satisfies = "✓" if sol.get('satisfies_targets') else "✗"
            
            f.write(f"  {i}. {method} ({cfg_str}) {satisfies}\n")
            f.write(f"     準確率: {obj['accuracy_change']*100:+.2f}%, ")
            f.write(f"GPU: {obj['gpu_peak_change']*100:+.2f}%, ")
            f.write(f"延遲: {obj['latency_change']*100:+.2f}%\n")
        f.write("\n")

    # 6. 推薦配置
    recommended = summary.get('recommended', {})
    if recommended and recommended.get('config'):
        f.write("【推薦的量化配置】\n")
        f.write(f"量化方法: {recommended.get('method', 'N/A')}\n")
        f.write(f"是否滿足所有目標: {'✓ 是' if recommended.get('satisfies_targets') else '✗ 否'}\n")
        f.write("\n")
        
        # 配置詳情
        config = recommended.get('config', {})
        f.write("配置參數:\n")
        for key, value in config.items():
            if key != 'modules_to_not_convert':  # 排除太長的字段
                display_key = key.replace('bnb_4bit_', '').replace('gptq_', '').replace('awq_', '')
                f.write(f"  {display_key}: {value}\n")
        f.write("\n")
        
        # 目標達成情況
        obj = recommended.get('objectives', {})
        f.write("目標達成情況:\n")
        acc_change = obj.get('accuracy_change', 0) * 100
        gpu_change = obj.get('gpu_peak_change', 0) * 100
        lat_change = obj.get('latency_change', 0) * 100
        
        acc_target = targets.get('accuracy_min', 0) * 100
        gpu_target = targets.get('gpu_peak_max', 0) * 100
        lat_target = targets.get('latency_max', 0) * 100
        
        acc_pass = "✓" if acc_change >= acc_target else "✗"
        gpu_pass = "✓" if gpu_change <= gpu_target else "✗"
        lat_pass = "✓" if lat_change <= lat_target else "✗"
        
        f.write(f"  準確率變化: {acc_change:+.2f}% (目標: ≥{acc_target:+.1f}%) {acc_pass}\n")
        f.write(f"  GPU 峰值變化: {gpu_change:+.2f}% (目標: ≤{gpu_target:+.1f}%) {gpu_pass}\n")
        f.write(f"  延遲變化: {lat_change:+.2f}% (目標: ≤{lat_target:+.1f}%) {lat_pass}\n")
        f.write("\n")
        
        # 推薦配置的絕對性能
        if 'aggregated' in recommended:
            agg = recommended['aggregated']
            f.write("推薦配置的絕對性能:\n")
            f.write(f"  準確率: {agg.get('accuracy', 0)*100:.2f}%\n")
            f.write(f"  GPU 峰值: {agg.get('gpu_peak_mb', 0):.0f} MB\n")
            f.write(f"  平均延遲: {agg.get('avg_latency_ms', 0):.2f} ms\n")
            f.write("\n")
        
        # 各資料集表現
        if 'per_dataset' in recommended:
            f.write("推薦配置在各資料集的表現:\n")
            for dataset, metrics in recommended['per_dataset'].items():
                f.write(f"  {dataset}:\n")
                f.write(f"    準確率: {metrics.get('accuracy', 0)*100:.2f}%\n")
                f.write(f"    GPU 峰值: {metrics.get('gpu_peak_mb', 0):.0f} MB\n")
                f.write(f"    樣本數: {metrics.get('num_samples', 'N/A')}\n")
            f.write("\n")
    else:
        f.write("【推薦的量化配置】\n")
        f.write("未找到滿足所有目標的配置\n")
        f.write("\n")
    
    f.write("\n" + "="*70 + "\n")
    f.write("報告結束\n")
    f.write("="*70 + "\n")

print(f"\n✓ 報告已匯出至: {report_file}")
print(f"\n【報告摘要】")
print(f"實驗: {summary.get('experiment_name', 'N/A')}")
print(f"試驗數: {opt_info.get('total_trials', 'N/A')}")
print(f"Pareto 解: {opt_info.get('pareto_solutions', 'N/A')}")
print(f"滿足目標: {opt_info.get('satisfying_solutions', 'N/A')}")

if recommended and recommended.get('config'):
    print(f"\n【推薦配置】")
    print(f"方法: {recommended.get('method', 'N/A')}")
    obj = recommended.get('objectives', {})
    print(f"準確率變化: {obj.get('accuracy_change', 0)*100:+.2f}%")
    print(f"GPU 峰值變化: {obj.get('gpu_peak_change', 0)*100:+.2f}%")
    print(f"延遲變化: {obj.get('latency_change', 0)*100:+.2f}%")
    print(f"滿足目標: {'✓ 是' if recommended.get('satisfies_targets') else '✗ 否'}")

print(f"\n所有視覺化完成！")


✓ 報告已匯出至: optimization/gemma-2-2b-it-multiobjective-opt_20260108_224919/visualization_report.txt

【報告摘要】
實驗: gemma-2-2b-it-multiobjective-opt
試驗數: 20
Pareto 解: 8
滿足目標: 2

【推薦配置】
方法: gptq
準確率變化: -1.69%
GPU 峰值變化: -53.69%
延遲變化: +47.34%
滿足目標: ✓ 是

所有視覺化完成！
